# FYP 2: Time-Series Data Extraction Pipeline
**Project:** Vision-based Movement Analysis of Rehabilitation Exercises

**Dataset:** REHAB24-6 (Addresses FYP1 comments on diverse lighting and clothing)

**Objective:** To extract frame-by-frame skeletal coordinate data using MediaPipe. To support our Ablation Study, we perform a **Dual Extraction**:
1. **Baseline Model:** Uses MediaPipe's built-in `smooth_landmarks=True`.
2. **Raw Model:** Uses `smooth_landmarks=False` to capture true tracking jitter for our custom filter evaluation.

In [3]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import glob
import zipfile

# --- 1. CONFIGURATION ---
dataset_root = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/zenodo/videos"
output_dir = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/FYP2_Data/MediaPipe_CSV"
os.makedirs(output_dir, exist_ok=True)

# --- 2. MEDIAPIPE INITIALIZATION ---
mp_pose = mp.solutions.pose

# The Baseline (MediaPipe's Internal Smoothing ON)
pose_baseline = mp_pose.Pose(
    static_image_mode=False, model_complexity=1,
    smooth_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5
)

# The Raw Extraction (Internal Smoothing OFF - For Custom Strict Filter)
pose_raw = mp_pose.Pose(
    static_image_mode=False, model_complexity=1,
    smooth_landmarks=False, min_detection_confidence=0.5, min_tracking_confidence=0.5
)

print("✅ Environment Configured. Baseline and Raw MediaPipe models initialized.")

✅ Environment Configured. Baseline and Raw MediaPipe models initialized.


I0000 00:00:1776576124.517362       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
I0000 00:00:1776576124.531062       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [13]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import glob

# --- 1. CONFIGURATION ---
dataset_root = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/zenodo/videos"
target_exercises = ["Ex1", "Ex2", "Ex3"]

output_dir = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/FYP2_Data/MediaPipe_CSV"
# We still create the main directory just in case it doesn't exist yet
os.makedirs(output_dir, exist_ok=True)

# Note: We assume pose_baseline and pose_raw are already initialized in the cell above!

print(f"🚀 Starting DUAL MediaPipe Extraction (Baseline & Raw) to {output_dir}...")

# --- 2. THE EXTRACTION LOOP ---
for exercise in target_exercises:
    
    # 🌟 THE FIX: Create a specific folder for each exercise (e.g., MediaPipe_CSV/Ex1)
    exercise_out_dir = os.path.join(output_dir, exercise)
    os.makedirs(exercise_out_dir, exist_ok=True)
    
    search_path = os.path.join(dataset_root, exercise, "*.mp4")
    video_files = glob.glob(search_path)
    
    print(f"📂 Processing {len(video_files)} videos in {exercise}...")
    
    for video_path in video_files:
        filename = os.path.basename(video_path)
        cap = cv2.VideoCapture(video_path)
        
        # We will keep two separate lists to save two CSVs per video
        baseline_data = []
        raw_data = []
        frame_idx = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            # Use 1-based indexing if Segmentation.csv uses 1-based indexing for 'first_frame'/'last_frame'
            frame_idx += 1 
            
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # --- EXTRACT BASELINE (MediaPipe Smoothing ON) ---
            res_base = pose_baseline.process(image_rgb)
            row_base = {'Frame': frame_idx, 'Extracted': False}
            if res_base.pose_landmarks:
                row_base['Extracted'] = True
                for i, lm in enumerate(res_base.pose_landmarks.landmark):
                    row_base[f'Lm{i}_x'] = lm.x
                    row_base[f'Lm{i}_y'] = lm.y
                    row_base[f'Lm{i}_v'] = lm.visibility
            baseline_data.append(row_base)
            
            # --- EXTRACT RAW (MediaPipe Smoothing OFF) ---
            res_raw = pose_raw.process(image_rgb)
            row_raw = {'Frame': frame_idx, 'Extracted': False}
            if res_raw.pose_landmarks:
                row_raw['Extracted'] = True
                for i, lm in enumerate(res_raw.pose_landmarks.landmark):
                    row_raw[f'Lm{i}_x'] = lm.x
                    row_raw[f'Lm{i}_y'] = lm.y
                    row_raw[f'Lm{i}_v'] = lm.visibility
            raw_data.append(row_raw)
                    
        cap.release()
        
        # Save the TWO CSV files PER VIDEO
        if baseline_data and raw_data:
            df_base = pd.DataFrame(baseline_data)
            df_raw = pd.DataFrame(raw_data)
            
            # 🌟 THE FIX: Save into the specific Ex1/Ex2/Ex3 folder instead of output_dir
            base_csv_name = os.path.join(exercise_out_dir, f"{filename.replace('.mp4', '_Baseline.csv')}")
            raw_csv_name = os.path.join(exercise_out_dir, f"{filename.replace('.mp4', '_Raw.csv')}")
            
            df_base.to_csv(base_csv_name, index=False)
            df_raw.to_csv(raw_csv_name, index=False)
            print(f"  ↳ ✅ Saved Baseline & Raw for {filename} ({len(df_base)} total frames)")
        else:
            print(f"❌ FAILED: {filename} (No frames readable)")

print("\n✨ DUAL MediaPipe Extraction Complete. CSVs are neatly organized by exercise!")

🚀 Starting DUAL MediaPipe Extraction (Baseline & Raw) to /Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/FYP2_Data/MediaPipe_CSV...
📂 Processing 26 videos in Ex1...
  ↳ ✅ Saved Baseline & Raw for PM_109-Camera17-30fps.mp4 (5226 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_114-Camera17-30fps.mp4 (5168 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_001-Camera17-30fps.mp4 (3016 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_002-Camera17-30fps.mp4 (1996 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_122-Camera18-30fps-transposed.mp4 (3739 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_012-Camera18-30fps-transposed.mp4 (2403 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_012-Camera17-30fps.mp4 (2403 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_032-Camera17-30fps.mp4 (2951 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_024-Camera17-30fps.mp4 (2104 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_101-Camera17-30fps.mp4 (1307 total frames)
  ↳ ✅ Saved Baseline & Raw for PM_039-Camer

## 4. YOLOv8n-Pose Extraction (Robustness Baseline)
**Addressing FYP1 Comments:** To justify the inclusion of YOLOv8, we extract pose data using `yolov8n-pose`. In the Analysis phase, we will compare these results against MediaPipe to highlight where YOLOv8 outperforms MediaPipe (e.g., heavy background clutter, occlusions) and note its limitations (e.g., lower keypoint precision compared to MediaPipe's 33 landmarks).

In [11]:
from ultralytics import YOLO
import cv2
import glob
import os
import pandas as pd

# --- 0. TARGET EXERCISES (STRICTLY EX1-EX3) ---
target_exercises = ["Ex1", "Ex2", "Ex3"]
dataset_root = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/zenodo/videos"

# --- 1. YOLO CONFIGURATION ---
yolo_model = YOLO('yolov8n-pose.pt') 
yolo_output_dir = "/Users/sofiasaifulrizal/Desktop/delta_2610/FYP2/FYP2_Data/YOLO_CSV"

print(f"🚀 Starting YOLOv8 Time-Series Extraction for {target_exercises}...\n")

# --- 2. YOLO EXTRACTION LOOP ---
for exercise in target_exercises:
    # 🌟 Create a specific folder for each exercise (e.g., YOLO_CSV/Ex1)
    exercise_out_dir = os.path.join(yolo_output_dir, exercise)
    os.makedirs(exercise_out_dir, exist_ok=True)
    
    video_files = glob.glob(os.path.join(dataset_root, exercise, "*.mp4"))
    
    print(f"📂 Processing {len(video_files)} videos in {exercise}...")
    
    for video_path in video_files:
        filename = os.path.basename(video_path)
        cap = cv2.VideoCapture(video_path)
        
        yolo_data = []
        frame_idx = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_idx += 1 
            
            # Get frame dimensions to normalize the coordinates
            h, w, _ = frame.shape
            
            # Run YOLOv8 inference (verbose=False keeps the console clean)
            results = yolo_model(frame, verbose=False)
            
            row_yolo = {'Frame': frame_idx, 'Extracted': False}
            
            # Check if a person/keypoints were detected
            if results[0].keypoints is not None and len(results[0].keypoints.xy[0]) > 0:
                row_yolo['Extracted'] = True
                
                # YOLOv8 returns 17 keypoints
                kpts = results[0].keypoints.xy[0].cpu().numpy()
                confs = results[0].keypoints.conf[0].cpu().numpy()
                
                for i, (kpt, conf) in enumerate(zip(kpts, confs)):
                    # Divide by width and height to convert to 0.0 - 1.0 scale
                    row_yolo[f'Ym{i}_x'] = kpt[0] / w
                    row_yolo[f'Ym{i}_y'] = kpt[1] / h
                    row_yolo[f'Ym{i}_v'] = conf
                    
            yolo_data.append(row_yolo)
                    
        cap.release()
        
        # Save Outputs safely into the specific Ex1/Ex2/Ex3 folder
        if yolo_data:
            csv_name = os.path.join(exercise_out_dir, f"{filename.replace('.mp4', '_YOLO.csv')}")
            pd.DataFrame(yolo_data).to_csv(csv_name, index=False)
            print(f"  ↳ ✅ Processed {filename} ({frame_idx} frames)")

print("\n✨ YOLOv8 Extraction Complete. Coordinates are normalized and neatly organized!")

🚀 Starting YOLOv8 Time-Series Extraction for ['Ex1', 'Ex2', 'Ex3']...

📂 Processing 26 videos in Ex1...
  ↳ ✅ Processed PM_109-Camera17-30fps.mp4 (5226 frames)
  ↳ ✅ Processed PM_114-Camera17-30fps.mp4 (5168 frames)
  ↳ ✅ Processed PM_001-Camera17-30fps.mp4 (3016 frames)
  ↳ ✅ Processed PM_002-Camera17-30fps.mp4 (1996 frames)
  ↳ ✅ Processed PM_122-Camera18-30fps-transposed.mp4 (3739 frames)
  ↳ ✅ Processed PM_012-Camera18-30fps-transposed.mp4 (2403 frames)
  ↳ ✅ Processed PM_012-Camera17-30fps.mp4 (2403 frames)
  ↳ ✅ Processed PM_032-Camera17-30fps.mp4 (2951 frames)
  ↳ ✅ Processed PM_024-Camera17-30fps.mp4 (2104 frames)
  ↳ ✅ Processed PM_101-Camera17-30fps.mp4 (1307 frames)
  ↳ ✅ Processed PM_039-Camera17-30fps.mp4 (3803 frames)
  ↳ ✅ Processed PM_101-Camera18-30fps-transposed.mp4 (1307 frames)
  ↳ ✅ Processed PM_002-Camera18-30fps-transposed.mp4 (1996 frames)
  ↳ ✅ Processed PM_122-Camera17-30fps.mp4 (3739 frames)
  ↳ ✅ Processed PM_039-Camera18-30fps-transposed.mp4 (3803 frames)
 